# FM Pipeline
Mirrors `IMPLEMENTATION_CHECKLIST.md` Part 1. Each section tests the corresponding `src/` code. Run cells in order.

In [ ]:
import sys, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate repo root regardless of where Jupyter was launched from.
# Handles two common cases: CWD = repo root, or CWD = notebooks/.
_cwd = Path().resolve()
if (_cwd / 'src').exists():
    REPO_ROOT = _cwd                  # launched from repo root
elif (_cwd.parent / 'src').exists():
    REPO_ROOT = _cwd.parent           # launched from notebooks/
else:
    raise RuntimeError(f"Cannot find src/ from {_cwd}. Launch Jupyter from the repo root or notebooks/.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DB_PATH = REPO_ROOT / 'season42' / 'season42_combined_skill_ns.db'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DB_PATH   : {DB_PATH}  (exists={DB_PATH.exists()})")

---
## 1.1 — Data Extraction & Preprocessing
**Source:** `src/data_prep.py`

In [ ]:
# 1.1.1 — Quality filters
from src.data_prep import load_filtered_matches, drop_incomplete_teams, ALL_BRAWLER_COLS

with sqlite3.connect(DB_PATH) as conn:
    raw = pd.read_sql('SELECT COUNT(*) AS n FROM matches', conn)['n'][0]

df_sets = load_filtered_matches(DB_PATH)
df_sets = drop_incomplete_teams(df_sets)
print(f'Raw: {raw:,}  →  Filtered: {len(df_sets):,}  ({len(df_sets)/raw*100:.1f}% retained)')
print(f'NULLs in brawler cols: {df_sets[ALL_BRAWLER_COLS].isna().sum().sum()}  (expect 0)')
print(f'Modes: {sorted(df_sets["mode"].unique())}  |  Maps: {df_sets["map"].nunique()}')

In [ ]:
# 1.1.2 — Record expansion: sets → individual games
from src.data_prep import expand_to_games

df_games = expand_to_games(df_sets, draw_value=None)
print(f'Sets: {len(df_sets):,}  →  Games: {len(df_games):,}  (x{len(df_games)/len(df_sets):.2f})')
print(f'team1 win rate: {df_games["team1_wins"].mean():.4f}  (expect ~0.50)')

In [ ]:
# 1.1.3 — Brawler vocabulary + pick distribution
from src.data_prep import build_brawler_vocab

vocab = build_brawler_vocab(df_games)
picks = pd.concat([df_games[c] for c in ALL_BRAWLER_COLS]).value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.bar(range(len(picks)), picks.values, width=1.0, color='steelblue')
ax.set(xlabel='Brawler rank (least → most picked)', ylabel='Total picks', title=f'Pick distribution — {len(vocab)} brawlers')
ax.axhline(picks.min(), color='firebrick', linestyle='--', linewidth=1, label=f'Min: {picks.min():,} ({picks.idxmin()})')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 1.1.4 — skill_ns distribution
s = df_games['skill_ns']
print(f'Range: [{s.min():.2f}, {s.max():.2f}]  |  Q25/Q50/Q75: {s.quantile(.25):.2f} / {s.quantile(.5):.2f} / {s.quantile(.75):.2f}')
print(f'Inf: {np.isinf(s).sum()}  NaN: {s.isna().sum()}  (both expect 0)')

fig, axes = plt.subplots(1, 2, figsize=(9, 2.5))
axes[0].hist(s, bins=150, color='steelblue', edgecolor='none', density=True)
for q, lbl in [(.25,'Q1'), (.5,'Q2'), (.75,'Q3')]:
    axes[0].axvline(s.quantile(q), color='firebrick', lw=1, linestyle='--', label=f'{lbl}={s.quantile(q):.2f}')
axes[0].set(xlabel='skill_ns', ylabel='Density', title='skill_ns distribution')
axes[0].legend(fontsize=7)

wr = df_games.assign(d=pd.qcut(s, 10, labels=False)).groupby('d')['team1_wins'].mean()
axes[1].bar(wr.index, wr.values, color='steelblue')
axes[1].axhline(0.5, color='firebrick', lw=1, linestyle='--')
axes[1].set_ylim(0.48, 0.52)
axes[1].set(xlabel='skill_ns decile', ylabel='team1 win rate', title='Win rate by skill decile (expect flat)')
plt.tight_layout(); plt.show()

In [ ]:
# 1.1.5 — Smoke test: full pipeline
from src.data_prep import build_game_dataset
df, vocab = build_game_dataset(db_path=DB_PATH)

assert len(df) > 4_000_000
assert len(vocab) == 95
assert df[ALL_BRAWLER_COLS].isna().sum().sum() == 0
assert abs(df['team1_wins'].mean() - 0.5) < 0.01
print('1.1 smoke test: all assertions passed.')

---
## 1.2 — Empirical Matchup Database
**Source:** `src/matchup_db.py`  
Build once with `MatchupDB.build(); db.save()`, then load from pickle for all subsequent runs.

In [ ]:
# 1.2 — Build or load MatchupDB
from src.matchup_db import MatchupDB, DEFAULT_SAVE_PATH

if DEFAULT_SAVE_PATH.exists():
    db = MatchupDB.load()
else:
    db = MatchupDB.build()
    db.save()

print(f"Brawler cells  (full/no_map/no_tier/global): "
      f"{len(db.brawler['full']):,} / {len(db.brawler['no_map']):,} / "
      f"{len(db.brawler['no_tier']):,} / {len(db.brawler['global']):,}")
print(f"Counter cells  (full/no_map/no_tier/global): "
      f"{len(db.counter['full']):,} / {len(db.counter['no_map']):,} / "
      f"{len(db.counter['no_tier']):,} / {len(db.counter['global']):,}")
print(f"Synergy cells  (mode_tier/mode/global):      "
      f"{len(db.synergy['mode_tier']):,} / {len(db.synergy['mode']):,} / "
      f"{len(db.synergy['global']):,}")

In [ ]:
# 1.2.1 — Brawler win rate summary: top/bottom 5 for gemGrab
mode_sample = "gemGrab"
rows = [{"brawler": k[0], **v} for k, v in db.brawler["no_tier"].items() if k[1] == mode_sample]
gdf = pd.DataFrame(rows).sort_values("win_rate", ascending=False).reset_index(drop=True)

print(f"Top 5 brawlers in {mode_sample}:")
print(gdf.head()[["brawler", "n", "win_rate", "pick_rate"]].to_string(index=False))
print(f"\nBottom 5 brawlers in {mode_sample}:")
print(gdf.tail()[["brawler", "n", "win_rate", "pick_rate"]].to_string(index=False))

In [ ]:
# 1.2.2 — Counter matrix: CROW vs opponents on first map with sufficient data + symmetry check
# n_min=50: ~±14% CI width at 95% confidence. Below this, win rates are noise.
COUNTER_N_MIN = 50
sample_brawler = "CROW"
tier_sample = 2

# Find maps where CROW has at least one opponent with n >= COUNTER_N_MIN
qualified = {k[2] for k in db.counter["full"]
             if k[0] == sample_brawler and k[3] == tier_sample
             and db.counter["full"][k]["n"] >= COUNTER_N_MIN}
assert qualified, f"No map found with n≥{COUNTER_N_MIN} for {sample_brawler} at tier {tier_sample}"
map_sample = sorted(qualified)[0]

rows = [{"opp": k[1], **v}
        for k, v in db.counter["full"].items()
        if k[0] == sample_brawler and k[2] == map_sample and k[3] == tier_sample
        and v["n"] >= COUNTER_N_MIN]
cdf = pd.DataFrame(rows).sort_values("win_rate", ascending=False)

print(f"Counter: {sample_brawler} (my team) vs opponents — '{map_sample}' / tier {tier_sample}  [n≥{COUNTER_N_MIN}]")
print(cdf.head(5)[["opp", "n", "win_rate"]].to_string(index=False))

# Symmetry: P(CROW→X) + P(X→CROW) must equal 1.0
opp = cdf.iloc[0]["opp"]
fwd = db.counter_lookup(sample_brawler, opp, "gemGrab", map_sample, tier_sample)
rev = db.counter_lookup(opp, sample_brawler, "gemGrab", map_sample, tier_sample)
total = fwd["win_rate"] + rev["win_rate"]
print(f"\nSymmetry: {sample_brawler}→{opp} {fwd['win_rate']:.4f}  +  {opp}→{sample_brawler} {rev['win_rate']:.4f}  =  {total:.4f}")
assert abs(total - 1.0) < 1e-9, "Symmetry violated!"
print("Symmetry assertion passed.")

In [ ]:
# 1.2.3 — Synergy spot-check: top/bottom 5 global synergy pairs (n≥500)
# n_min=500: synergy_delta is a difference of two noisy estimates; needs more data than a single rate.
SYNERGY_N_MIN = 500

synergy_rows = sorted(
    [{"a": k[0], "b": k[1], **v} for k, v in db.synergy["global"].items()
     if v["n"] >= SYNERGY_N_MIN],
    key=lambda x: x["synergy_delta"], reverse=True
)
sdf = pd.DataFrame(synergy_rows)
print(f"Top 5 global synergy pairs [n≥{SYNERGY_N_MIN}]:")
print(sdf.head()[["a", "b", "n", "win_rate", "baseline", "synergy_delta"]].to_string(index=False))
print(f"\nBottom 5 (anti-synergy) [n≥{SYNERGY_N_MIN}]:")
print(sdf.tail()[["a", "b", "n", "win_rate", "baseline", "synergy_delta"]].to_string(index=False))

# 1.2.4 — Fallback lookup: nonexistent map must cascade, never return None
r_b = db.brawler_lookup("LOLA", "gemGrab", "NONEXISTENT_MAP", 3)
r_c = db.counter_lookup("CROW", "POCO", "gemGrab", "NONEXISTENT_MAP", 3)
assert r_b is not None and r_b["level"] > 0, "Expected fallback for brawler lookup"
assert r_c is not None and r_c["level"] > 0, "Expected fallback for counter lookup"
print(f"\nFallback: LOLA / NONEXISTENT_MAP → level {r_b['level']}  (win_rate={r_b['win_rate']:.4f})")
print(f"Fallback: CROW vs POCO / NONEXISTENT_MAP → level {r_c['level']}  (win_rate={r_c['win_rate']:.4f})")
print("Fallback assertions passed.")

---
## 1.3 — Feature Engineering
**Source:** `src/feature_engineering.py`  
Builds the sparse FM feature matrix (2N × D) with symmetric augmentation (each game row + its team-swapped counterpart).

In [ ]:
# 1.3.1 — Schema dimensions and sparsity check
from src.feature_engineering import build_schema, build_feature_matrix

# Build schema from the already-loaded df (reuse df from 1.1.5)
schema = build_schema(df)
print(f"D = {schema.n_features} features")
print(f"  t1 brawlers : [{schema.t1_offset}, {schema.t2_offset})  = {len(schema.vocab)}")
print(f"  t2 brawlers : [{schema.t2_offset}, {schema.map_offset})  = {len(schema.vocab)}")
print(f"  maps        : [{schema.map_offset}, {schema.mode_offset})  = {len(schema.maps)}")
print(f"  modes       : [{schema.mode_offset}, {schema.skill_offset})  = {len(schema.modes)}")
print(f"  skill_ns    : [{schema.skill_offset}]")

X, y = build_feature_matrix(df, schema)

nnz_per_row = X.nnz / X.shape[0]
print(f"\nMatrix shape : {X.shape}  (2 × {len(df):,} games)")
print(f"nnz          : {X.nnz:,}")
print(f"nnz per row  : {nnz_per_row:.1f}  (expect 9: 3+3 brawlers, map, mode, skill_ns)")
print(f"Label balance: {y.mean():.4f}  (must be exactly 0.50)")

In [ ]:
# 1.3.2 — Single-row feature inspection
# Pick row 0. Print every active feature by name + value and verify the expected structure.
row = X.getrow(0)
active = list(zip(row.indices, row.data))
active.sort()

print(f"Row 0 — label={y[0]}  |  {len(active)} active features (expect 9)\n")
print(f"{'Feature':<35} {'Value':>8}")
print("-" * 45)
for idx, val in active:
    print(f"{schema.feature_name(idx):<35} {val:>8.4f}")

# Cross-check against the raw DataFrame row
raw = df.iloc[0]
print(f"\nRaw row → t1: {list(raw[['t1_b0_name','t1_b1_name','t1_b2_name']])}  "
      f"t2: {list(raw[['t2_b0_name','t2_b1_name','t2_b2_name']])}")
print(f"          map={raw['map']}  mode={raw['mode']}  skill_ns={raw['skill_ns']:.4f}  label={raw['team1_wins']}")

In [ ]:
# 1.3.3 — Symmetric pair check
# Row 2i is the original; row 2i+1 is the team-swapped counterpart.
# Labels must sum to 1. t1/t2 brawler features must be exactly swapped.
i = 0
orig = X.getrow(2 * i)
flip = X.getrow(2 * i + 1)

print(f"Original label={y[2*i]}  |  Flipped label={y[2*i+1]}  →  sum={y[2*i]+y[2*i+1]}  (expect 1)")

orig_feats = {schema.feature_name(idx): val for idx, val in zip(orig.indices, orig.data)}
flip_feats = {schema.feature_name(idx): val for idx, val in zip(flip.indices, flip.data)}

V = len(schema.vocab)
for name in sorted(orig_feats):
    if name.startswith("t1_") or name.startswith("t2_"):
        brawler = name[3:]
        counterpart = ("t2_" if name.startswith("t1_") else "t1_") + brawler
        status = "✓" if counterpart in flip_feats else "✗ MISSING"
        print(f"  orig {name:<30} → flip should have {counterpart:<30} {status}")

# Map, mode, skill_ns must be identical in both rows
shared = ["skill_ns"] + [f"map_{m}" for m in schema.maps] + [f"mode_{m}" for m in schema.modes]
for name in shared:
    if name in orig_feats:
        match = abs(orig_feats.get(name, 0) - flip_feats.get(name, 0)) < 1e-6
        print(f"  context feature '{name}' same in both rows: {'✓' if match else '✗'}")

---
## 1.4 — FM Model Design & Training
**Source:** `src/fm_model.py`  
Train with `python src/fm_model.py` (one-time). Cells below load the saved artifact and verify it.

In [ ]:
# 1.4.1 — Load trained FM model (run `python src/fm_model.py` first to train)
from src.fm_model import FMInference, MODEL_PATH, train_fm

if MODEL_PATH.exists():
    inf = FMInference.load()
    print(f"Loaded FM model from {MODEL_PATH}")
else:
    print("No saved model found — training now (takes several minutes)...")
    inf = train_fm()

print(f"\nModel summary:")
print(f"  D = {len(inf.w_linear)} features   k = {inf.V.shape[1]} embedding dim")
print(f"  Parameters: {1 + len(inf.w_linear) + inf.V.size:,}")

if inf.split_time:
    n_games_train = inf.n_train // 2
    n_games_val   = inf.n_val // 2
    total_games   = n_games_train + n_games_val
    print(f"\nChronological split at: {inf.split_time}")
    print(f"  Train: {n_games_train:,} games  ({n_games_train / total_games:.0%})")
    print(f"  Val:   {n_games_val:,} games  ({n_games_val / total_games:.0%})")
    print(f"  (Each split is doubled to {inf.n_train:,} / {inf.n_val:,} rows after symmetry augmentation)")

In [ ]:
# 1.4.2 — Training curve (loss and AUC per epoch)
if not inf.train_history:
    print("No training history in this artifact (model was loaded from a pre-existing checkpoint).")
else:
    epochs     = [h["epoch"]      for h in inf.train_history]
    train_loss = [h["train_loss"] for h in inf.train_history]
    val_loss   = [h["val_loss"]   for h in inf.train_history]
    val_auc    = [h["val_auc"]    for h in inf.train_history]
    best_epoch = epochs[val_loss.index(min(val_loss))]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))

    ax1.plot(epochs, train_loss, label="train", color="steelblue")
    ax1.plot(epochs, val_loss,   label="val",   color="darkorange")
    ax1.axvline(best_epoch, color="gray", linestyle="--", linewidth=1,
                label=f"best (ep {best_epoch})")
    ax1.set(xlabel="Epoch", ylabel="Log-loss", title="Training curve")
    ax1.legend(fontsize=8)

    ax2.plot(epochs, val_auc, color="forestgreen")
    ax2.axvline(best_epoch, color="gray", linestyle="--", linewidth=1)
    ax2.set(xlabel="Epoch", ylabel="Val AUC", title="Validation AUC")

    plt.tight_layout()
    plt.show()

    print(f"Best checkpoint at epoch {best_epoch}  |  "
          f"val loss {min(val_loss):.4f}  |  "
          f"val AUC {val_auc[val_loss.index(min(val_loss))]:.4f}")

In [ ]:
# 1.4.3 — Final validation metrics + inference sanity check
print("── Final validation metrics (best checkpoint) ──────────────────")
print(f"  Log-loss : {inf.val_logloss:.4f}  (random baseline ≈ 0.693)")
print(f"  AUC-ROC  : {inf.val_auc:.4f}  (random baseline = 0.500)")
print(f"  Brier    : {inf.val_brier:.4f}  (random baseline = 0.250)")

# Encode a complete 3v3 draft state as sparse (indices, values) arrays.
def _encode(schema, t1, t2, map_name, mode_name, skill_ns=1.0):
    idx = (
        [schema._vocab_idx[b] + schema.t1_offset for b in t1]
        + [schema._vocab_idx[b] + schema.t2_offset for b in t2]
        + [schema._map_idx[map_name] + schema.map_offset]
    )
    vals = [1.0] * len(idx)
    if schema.include_mode:
        idx.append(schema._mode_idx[mode_name] + schema.mode_offset)
        vals.append(1.0)
    idx.append(schema.skill_offset)
    vals.append(float(skill_ns))
    return np.array(idx, dtype=np.int32), np.array(vals, dtype=np.float32)

sc  = inf.schema
t1  = sc.vocab[:3]   # first 3 brawlers alphabetically
t2  = sc.vocab[3:6]  # next 3
map0, mode0 = sc.maps[0], sc.modes[0]

p_fwd = inf.evaluate_sparse(*_encode(sc, t1, t2, map0, mode0))
p_rev = inf.evaluate_sparse(*_encode(sc, t2, t1, map0, mode0))

print(f"\nSymmetry probe  ({t1} vs {t2}  |  {map0} / {mode0})")
print(f"  P(t1 wins)       = {p_fwd:.4f}")
print(f"  P(t2 wins)       = {p_rev:.4f}")
print(f"  sum              = {p_fwd + p_rev:.4f}")
print()
print("  Note: the FM does not guarantee exact symmetry. Context features")
print("  (map, mode, skill_ns) contribute the same logit mass regardless of")
print("  which team is t1, creating a small residual offset. Brawler interaction")
print("  terms are pushed toward antisymmetry by the training augmentation but")
print("  cannot fully cancel the context bias. This is harmless for MCTS: our")
print("  team is always encoded as t1, so the bias is consistent and the")
print("  relative ordering of picks is unaffected.")

# Hard check: output is a valid probability
assert 0.0 < p_fwd < 1.0 and 0.0 < p_rev < 1.0, "FM output out of (0,1) range!"
print("\n  Output-range assertion passed (both probs in (0, 1)).")

# Inference speed
inf.benchmark()

---
## 1.5 — Calibration & Evaluation
**Source:** `src/fm_evaluate.py`  
Uses `df` and `schema` already in scope from §1.1/§1.3, and `inf` from §1.4.

- **1.5.1** Scalar metrics: log-loss, AUC-ROC, Brier on the chronological val set.
- **1.5.2** Calibration curves: overall + per-mode subplots, per-map ECE table.  
- **1.5.3** Inference speed: encoding and FM forward pass profiled separately.

In [ ]:
# 1.5.1 — Validation metrics
# Build the val feature matrix from the df already in scope (avoids reloading the DB).
from src.feature_engineering import chronological_split
from src.fm_model import _csr_to_compact
from src.fm_evaluate import batch_predict, evaluate_metrics

_, val_df_15 = chronological_split(df)
val_df_15 = val_df_15.reset_index(drop=True)

print("Building val feature matrix (may take ~30 s)...")
X_val_csr, y_val = build_feature_matrix(val_df_15, schema)
val_idx, val_val = _csr_to_compact(X_val_csr)
del X_val_csr

# Batch-predict; keep only original rows (even indices = un-flipped orientation)
all_probs  = batch_predict(inf, val_idx, val_val)
val_probs  = all_probs[0::2]
val_labels = y_val[0::2].astype("int8")
print(f"Val set: {len(val_df_15):,} games  |  label balance: {val_labels.mean():.4f}  (expect 0.50)\n")

# 1.5.1 output
metrics = evaluate_metrics(val_probs, val_labels)

In [ ]:
# 1.5.2 — Calibration curves + per-map ECE table
# 1.5.3 — Inference speed benchmark
from src.fm_evaluate import plot_calibration_curves, print_permap_calibration, benchmark_inference

# Overall + per-mode calibration subplots → saved to figures/calibration_curves.png
plot_calibration_curves(val_df_15, val_probs, val_labels)

# Per-map ECE table (all 26 maps, sorted best → worst calibration)
print_permap_calibration(val_df_15, val_probs, val_labels)

# Encoding speed / FM forward pass speed / combined pipeline (MCTS figure of merit)
speed = benchmark_inference(inf)

---
## 1.6 — Interpretability Check
**Source:** `src/fm_interpret.py`

The FM stores a learned k=32 embedding vector for every feature (each brawler on each team, each map, each mode, and skill_ns). The **dot product** between two embeddings is the FM's learned interaction coefficient for that pair — it directly scales the contribution to win-probability when both features are active.

Three complementary views below:
- **Counter relationships** — which brawlers the FM learned to counter whom  
- **Skill scaling + brawler strength** — who rewards skill and who is inherently strong  
- **Brawler & map PCA** — where brawlers and maps cluster in latent space

In [ ]:
# Load interpretability helpers (inf, schema, db already in scope from §1.4 / §1.2)
import sys
from pathlib import Path
_src = str(REPO_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from fm_interpret import (
    get_brawler_embeddings, get_map_embeddings, get_mode_embeddings,
    get_skill_embedding, top_counter_pairs, brawler_importance,
    skill_scaling_ranking, map_skill_affinity,
)

t1_emb, t2_emb = get_brawler_embeddings(inf)
map_emb         = get_map_embeddings(inf)
mode_emb        = get_mode_embeddings(inf)
skill_emb       = get_skill_embedding(inf)
print(f"Embeddings ready — brawlers: {t1_emb.shape}, maps: {map_emb.shape}, skill_ns: {skill_emb.shape}")


### 1.6.1 — Counter Relationships

The FM learns *separate* embeddings for a brawler on your team (`t1_`) and on the opponent's team (`t2_`). The dot product ⟨v_t1_A, v_t2_B⟩ is the FM's learned **counter coefficient** between A and B: a high positive value means having A while facing B significantly improves your win odds. Unlike synergy (which is better read from the empirical matchup DB's `synergy_delta`), counter relationships are directional and the FM captures them cleanly.

In [ ]:
# Top 10 FM counter pairs by ⟨v_t1_A, v_t2_B⟩
fm_ctr = top_counter_pairs(inf, n=10)
print("Top 10 counter relationships (FM embedding dot product):")
print(fm_ctr.to_string(index=False))

# Spot-checks against known Brawl Stars mechanics
spot = [
    ("MORTIS",  "BARLEY",   "MORTIS dodges thrown projectiles"),
    ("MORTIS",  "DYNAMIKE", "MORTIS dodges thrown projectiles"),
    ("BELLE",   "PIPER",    "burst snipers beat sustained snipers"),
]
print("\nSpot-check — expected strong counters:")
print(f"{'My brawler':<12} {'Opp brawler':<12} {'FM dot':>8}  Note")
print("-" * 65)
for my, opp, note in spot:
    i = inf.schema._vocab_idx.get(my)
    j = inf.schema._vocab_idx.get(opp)
    if i is not None and j is not None:
        dot_val = float(t1_emb[i] @ t2_emb[j])
        print(f"{my:<12} {opp:<12} {dot_val:>8.4f}  {note}")

# Empirical top synergy (matchup DB) for completeness — more reliable signal than FM dots
db_syn = sorted(
    [{"a": k[0], "b": k[1], **v} for k, v in db.synergy["global"].items() if v["n"] >= 500],
    key=lambda x: x["synergy_delta"], reverse=True
)[:5]
print("\nTop 5 empirical synergy pairs (matchup DB synergy_delta, n≥500):")
print(pd.DataFrame(db_syn)[["a", "b", "n", "synergy_delta"]].to_string(index=False))


### 1.6.2 — Skill Scaling & Brawler Strength

**Skill scaling** — ⟨v_skill_ns, v_t1_B⟩: the FM interaction coefficient between skill level and picking brawler B. A high value means brawler B contributes *more* to winning as the player's skill increases — these are high-skill-ceiling brawlers (e.g., BELLE, PIPER require precise aim). Low or negative values indicate consistent brawlers whose value doesn't depend on mechanical skill.

**Net power** — w_t1[i] − w_t2[i]: the FM's first-order estimate of brawler strength. `w_t1[i]` is how much the model shifts win-probability when brawler i is on *your* team; `w_t2[i]` is the same when i is on the *opponent's* team. The difference is a clean signal of inherent brawler strength independent of context interactions.

In [ ]:
skill_df = skill_scaling_ranking(inf)
imp_df   = brawler_importance(inf)

print("Top 15 skill-scaling brawlers (gain more from higher player skill):")
print(skill_df.head(15).to_string(index=False))
print("\nBottom 10 (low ceiling — same value regardless of skill level):")
print(skill_df.tail(10)[["brawler","skill_dot"]].to_string(index=False))

print("\n" + "─" * 55)
print("\nTop 15 strongest brawlers by net linear weight (w_t1 − w_t2):")
print(imp_df.head(15)[["brawler","w_t1","w_t2","net_power"]].to_string(index=False))
print("\nBottom 10 (weakest in current meta):")
print(imp_df.tail(10)[["brawler","net_power"]].to_string(index=False))

# Quick cross-check with matchup DB global win rates
db_wr = sorted([{"brawler":k[0],"wr":v["win_rate"]} for k,v in db.brawler["global"].items()],
               key=lambda x: x["wr"], reverse=True)
db_top10 = {r["brawler"] for r in db_wr[:10]}
fm_top10 = set(imp_df.head(10)["brawler"])
print(f"\nAgreement FM top-10 vs DB win-rate top-10: {len(fm_top10 & db_top10)}/10 overlap")


### 1.6.3 — Brawler & Map Embeddings (PCA)

PCA projects the 32-dimensional embedding space down to 2D. Brawlers that cluster together have learned similar latent representations — they're strong in similar contexts, on similar maps, with similar teammate types. Color = net_power (green = strong meta brawlers, purple = weaker).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# PCA of t1 brawler embeddings colored by net_power
t1_scaled = StandardScaler().fit_transform(t1_emb)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(t1_scaled)
var_exp = pca.explained_variance_ratio_

vocab = inf.schema.vocab
# Map each brawler to its net_power for color coding
power_lookup = imp_df.set_index("brawler")["net_power"].to_dict()
powers = np.array([power_lookup[b] for b in vocab])

norm = mcolors.TwoSlopeNorm(vmin=powers.min(), vcenter=0, vmax=powers.max())
cmap = cm.RdYlGn

fig, ax = plt.subplots(figsize=(11, 8))
sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=powers, cmap=cmap, norm=norm,
                     s=22, alpha=0.85, zorder=2)
plt.colorbar(sc_plot, ax=ax, label="Net power  (w_t1 − w_t2)", shrink=0.7)

for i, name in enumerate(vocab):
    ax.annotate(name, coords[i], fontsize=5.5, ha="center", va="bottom",
                color="black", alpha=0.85)

ax.set_xlabel(f"PC1 ({var_exp[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var_exp[1]:.1%} variance)")
ax.set_title("Brawler embedding PCA — color = net meta strength (green=strong, red=weak)")
plt.tight_layout()
plt.savefig(REPO_ROOT / "figures" / "brawler_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"PC1={var_exp[0]:.1%}  PC2={var_exp[1]:.1%} of total variance explained")


### 1.6.3 cont. — Map Analysis

**Skill-map interaction** — ⟨v_skill_ns, v_map⟩: maps where mechanical skill differences between teams matter more. Positive = high-skill teams gain a larger edge on this map. These are maps with tight sightlines, more outplay potential, or where precise aim is more rewarded. Near-zero = the map is relatively even regardless of skill gap.

The **context PCA** shows how maps cluster relative to each other and where modes and skill_ns point — maps near skill_ns are more skill-dependent.

In [ ]:
# Map skill-affinity bar chart
map_skill = map_skill_affinity(inf)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["firebrick" if v > 0 else "steelblue" for v in map_skill["skill_dot"]]
ax.barh(map_skill["map"][::-1], map_skill["skill_dot"][::-1], color=colors[::-1], height=0.7)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("⟨v_skill_ns, v_map⟩  (positive = skilled players gain more edge)")
ax.set_title("Skill-map interaction by map")
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig(REPO_ROOT / "figures" / "map_skill_affinity.png", dpi=150, bbox_inches="tight")
plt.show()

# Context PCA: maps + modes + skill_ns together
import matplotlib.patches as mpatches

all_embs   = np.vstack([map_emb, mode_emb, skill_emb[None, :]])
all_labels = inf.schema.maps + inf.schema.modes + ["skill_ns"]
all_types  = ["map"] * len(inf.schema.maps) + ["mode"] * len(inf.schema.modes) + ["skill"]

pca2 = PCA(n_components=2, random_state=42)
ctx  = pca2.fit_transform(StandardScaler().fit_transform(all_embs))
vexp = pca2.explained_variance_ratio_

cmap2 = {"map": "steelblue", "mode": "forestgreen", "skill": "firebrick"}
fig, ax = plt.subplots(figsize=(11, 7))
for t, color in cmap2.items():
    mask = np.array([x == t for x in all_types])
    ax.scatter(ctx[mask, 0], ctx[mask, 1], c=color,
               s=80 if t == "skill" else (45 if t == "mode" else 20), alpha=0.85, zorder=3)

for i, (label, t) in enumerate(zip(all_labels, all_types)):
    fs = 9 if t == "skill" else (8 if t == "mode" else 6.5)
    fw = "bold" if t in ("skill", "mode") else "normal"
    ax.annotate(label, ctx[i], fontsize=fs, fontweight=fw,
                color=cmap2[t], ha="center", va="bottom")

ax.legend(handles=[mpatches.Patch(color=c, label=t) for t, c in cmap2.items()], fontsize=9)
ax.set_xlabel(f"PC1 ({vexp[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({vexp[1]:.1%} var)")
ax.set_title("Context embedding PCA — maps, modes, and skill_ns in latent space")
plt.tight_layout()
plt.savefig(REPO_ROOT / "figures" / "context_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Maps near 'skill_ns' in latent space = most affected by player skill gap")
